### Import Libraries

In [1]:
import re
import contractions
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

### Config

In [2]:
# Constants
RANDOM_STATE = 42
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
TARGET_COL = "label"
TEXT_COL = "text"

# Base path
BASE_DIR = Path.cwd().resolve().parents[1]

# Paths
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DATA_PATH = PROCESSED_DIR / "cleaned_sms.csv"

### Load Dataset

In [3]:
df = pd.read_csv(DATA_URL, sep="\t", header=None, names=[TARGET_COL, TEXT_COL])
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)

Shape: (5572, 2)

Data Types:
 label    object
text     object
dtype: object


In [4]:
# Preview
df.head(5)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Check Missing Values

In [5]:
# Check for nulls across all columns
df.isna().sum()

label    0
text     0
dtype: int64

**Observation: Missing Values**
- No missing values found in text or label columns
- No immediate handling required

### Check Duplicates

In [6]:
# Check duplicate rows
df.duplicated().sum()

403

**Observation: Duplicate Rows**
- Duplicate rows are present in the dataset
- These can bias the model by over-representing certain samples
- Will handle in preprocessing stage

### Target Distribution

In [7]:
df[TARGET_COL].value_counts(normalize=True)

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

**Observation: Class Distribution**
- Dataset is imbalanced (spam is a minority class)
- Accuracy alone may be misleading
- Will consider metrics like F1-score and possible class weighting

### Basic Text Inspection

#### 1. Random samples

In [8]:
df.sample(5, random_state=RANDOM_STATE)

,label,text
3245,ham,Squeeeeeze!! This is christmas hug.. If u lik ...
944,ham,And also I've sorta blown him off a couple tim...
1044,ham,Mmm thats better now i got a roast down me! i...
2484,ham,Mm have some kanji dont eat anything heavy ok
812,ham,So there's a ring that comes with the guys cos...


In [9]:
for _, row in df.sample(5).iterrows():
    print(row[TEXT_COL])
    print("-" * 60)

I told that am coming on wednesday.
------------------------------------------------------------
Haf u found him? I feel so stupid da v cam was working.
------------------------------------------------------------
Where are you ? What do you do ? How can you stand to be away from me ? Doesn't your heart ache without me ? Don't you wonder of me ? Don't you crave me ?
------------------------------------------------------------
You have an important customer service announcement. Call FREEPHONE 0800 542 0825 now!
------------------------------------------------------------
Yeah, don't go to bed, I'll be back before midnight
------------------------------------------------------------


#### 2. Text Length Analysis

In [10]:
# Length of each message
df["text_length"] = df[TEXT_COL].str.len()
df["text_length"].describe()

count    5572.000000
mean       80.489950
std        59.942907
min         2.000000
25%        36.000000
50%        62.000000
75%       122.000000
max       910.000000
Name: text_length, dtype: float64

In [11]:
df.groupby(TARGET_COL)["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4825.0,71.482487,58.440652,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.670683,28.873603,13.0,133.0,149.0,157.0,223.0


**Observation: Text Length**
- `spam` messages are generally longer than `ham `messages, as seen from the higher mean and median lengths. This is likely because spam messages often contain promotional or informational content
- Although there is some overlap in distributions, text length can serve as a useful feature for distinguishing between `spam` and `ham` messages

#### 3. Inspect Short vs Long Text

In [12]:
# Very short texts
df.sort_values("text_length").head(5)

,label,text,text_length
4498,ham,Ok,2
3051,ham,Ok,2
1925,ham,Ok,2
5357,ham,Ok,2
3833,ham,Ok.,3


In [13]:
# Very long texts
df.sort_values("text_length", ascending=False).head(5)

,label,text,text_length
1085,ham,For me the love should start with attraction.i...,910
1863,ham,The last thing i ever wanted to do was hurt yo...,790
2434,ham,Indians r poor but India is not a poor country...,629
1579,ham,How to Make a girl Happy? It's not at all diff...,611
2158,ham,Sad story of a Man - Last week was my b'day. M...,588


#### 4. Empty / Suspicious Text

In [14]:
df[df["text_length"] <= 2]

,label,text,text_length
1925,ham,Ok,2
3051,ham,Ok,2
4498,ham,Ok,2
5357,ham,Ok,2


**Observations:**
1. Raw Text Characteristics
    - Very short messages (e.g., "Ok") are present and are predominantly `ham`, indicating low-information conversational responses
    - Very long messages are also primarily `ham`, suggesting that length alone is not sufficient to classify `spam`, especially for extreme cases
    - This indicates that while spam messages are longer on average, text length should be used in combination with other features rather than as a standalone signal

2. Edge Cases
- Short texts may contribute little to model learning and could be considered for filtering or special handling
- Long texts may introduce noise or bias if not handled properly, especially in models sensitive to input length

### Motivation for Text Cleaning

The dataset contains raw text with HTML content, URLs, contractions, and inconsistent formatting

From inspection:
- HTML entities and formatting artifacts are present
- URLs and special tokens appear in some messages
- Contractions (e.g., "I've", "don't") may obscure important signals like negation

To prepare the data for downstream NLP tasks such as vectorization, we perform basic text cleaning to:
- Remove structural noise (HTML, scripts, URLs)
- Normalize text (expand contractions)
- Standardize formatting (whitespace cleanup)

The goal is to clean the data without aggressively removing useful semantic information, keeping the pipeline flexible for task-specific preprocessing later.

### Text Cleaning Function

In [15]:
def clean_text(text):
    if not isinstance(text, str):
        return None
        
    # --- HTML CLEANING ---
    # Parse HTML and remove unwanted tags (script/style contain noise like JS/CSS) 
    soup = BeautifulSoup(text, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()

    # Tags removed, Content retained with spaces
    text = soup.get_text(separator=" ")

    # --- URL REMOVAL ---
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # --- DECONTRACTION ---
    text = contractions.fix(text)

    # --- WHITESPACE CLEANUP ---
    text = re.sub(r"\s+", " ", text).strip()

    # --- FINAL VALIDATION ---
    if not text:
        return None
        
    return text

### Apply Cleaning to Dataset

In [16]:
# Apply cleaning function to create a normalized text column
# This will be used for all downstream preprocessing and modeling steps
df["clean_text"] = df[TEXT_COL].apply(clean_text)

### Post-Cleaning Validation & Analysis

#### 1. Qualitative Check: Raw vs Cleaned

In [17]:
sample_df = df.sample(5, random_state=RANDOM_STATE)
sample_df[[TEXT_COL, "clean_text"]]

,text,clean_text
3245,Squeeeeeze!! This is christmas hug.. If u lik ...,Squeeeeeze!! This is christmas hug.. If you li...
944,And also I've sorta blown him off a couple tim...,And also I have sorta blown him off a couple t...
1044,Mmm thats better now i got a roast down me! i...,Mmm that is better now i got a roast down me! ...
2484,Mm have some kanji dont eat anything heavy ok,Mm have some kanji do not eat anything heavy ok
812,So there's a ring that comes with the guys cos...,So there is a ring that comes with the guys co...


In [18]:
for _, row in sample_df.iterrows():
    print("RAW   :", row[TEXT_COL])
    print("CLEAN :", row["clean_text"])
    print("-" * 60)

RAW   : Squeeeeeze!! This is christmas hug.. If u lik my frndshp den hug me back.. If u get 3 u r cute:) 6 u r luvd:* 9 u r so lucky;) None? People hate u:
CLEAN : Squeeeeeze!! This is christmas hug.. If you lik my frndshp den hug me back.. If you get 3 you r cute:) 6 you r luvd:* 9 you r so lucky;) None? People hate you:
------------------------------------------------------------
RAW   : And also I've sorta blown him off a couple times recently so id rather not text him out of the blue looking for weed
CLEAN : And also I have sorta blown him off a couple times recently so id rather not text him out of the blue looking for weed
------------------------------------------------------------
RAW   : Mmm thats better now i got a roast down me! id b better if i had a few drinks down me 2! Good indian?
CLEAN : Mmm that is better now i got a roast down me! id b better if i had a few drinks down me 2! Good indian?
------------------------------------------------------------
RAW   : Mm have s

#### 2. Transformation Summary

In [19]:
print("Total rows:", len(df))

changed_mask = df[TEXT_COL].fillna("") != df["clean_text"].fillna("")
changed_count = changed_mask.sum()

print("Null cleaned_text:", df["clean_text"].isna().sum())
print("Changed rows:", changed_count)
print("Changed rows %:", changed_count / len(df))

Total rows: 5572
Null cleaned_text: 0
Changed rows: 2841
Changed rows %: 0.5098707824838478


#### 3. Inspect Changed Rows

In [20]:
df_changed = df.loc[changed_mask, [TEXT_COL, "clean_text"]]

for _, row in df_changed.sample(5, random_state=RANDOM_STATE).iterrows():
    print("RAW   :", row[TEXT_COL])
    print("CLEAN :", row["clean_text"])
    print("-" * 60)

RAW   : Good Morning my Dear........... Have a great &amp; successful day.
CLEAN : Good Morning my Dear........... Have a great & successful day.
------------------------------------------------------------
RAW   : Do you know why god created gap between your fingers..? So that, One who is made for you comes &amp; fills those gaps by holding your hand with LOVE..!
CLEAN : Do you know why god created gap between your fingers..? So that, One who is made for you comes & fills those gaps by holding your hand with LOVE..!
------------------------------------------------------------
RAW   : Love you aathi..love u lot..
CLEAN : Love you aathi..love you lot..
------------------------------------------------------------
RAW   : Just wanted to say holy shit you guys weren't kidding about this bud
CLEAN : Just wanted to say holy shit you guys were not kidding about this bud
------------------------------------------------------------
RAW   : Solve d Case : A Man Was Found Murdered On  &lt;DECIMAL

#### 4. Length Analysis

In [21]:
df["clean_length"] = df["clean_text"].str.len()

pd.DataFrame({
    "raw": df["text_length"].describe(),
    "clean": df["clean_length"].describe()
})

,raw,clean
count,5572.000000,5572.000000
mean,80.489950,81.221464
std,59.942907,59.984206
min,2.000000,2.000000
25%,36.000000,36.000000
50%,62.000000,62.000000
75%,122.000000,123.000000
max,910.000000,911.000000


#### 5. Edge Case Analysis

In [22]:
# Very short texts
short_mask = df["clean_text"].str.len() < 5
print("Very short texts (<5 chars):", short_mask.sum())

Very short texts (<5 chars): 17


In [23]:
df.loc[short_mask, [TEXT_COL, "clean_text"]].head(5)

,text,clean_text
261,Yup,Yup
287,Ok..,Ok..
1612,645,645
1925,Ok,Ok
2182,Ok.,Ok.


In [24]:
# Aggressive reduction check
reduction_mask = df["clean_length"] < df["text_length"] * 0.3
print("Significant reduction (<30%):", reduction_mask.sum())

Significant reduction (<30%): 0


**Observation: Post-Cleaning Validation**
- No invalid outputs were produced during cleaning, indicating that the cleaning function is robust and handles edge cases effectively
- Cleaning affected approximately half of the dataset, indicating meaningful normalization without being overly aggressive
- Contractions are correctly expanded (e.g., "I've" → "I have"), improving clarity and exposing negation for downstream tasks
- HTML entities are decoded during parsing (e.g., &amp; → &), while encoded placeholders like <DECIMAL> remain as text
- Numeric values and punctuation are preserved, retaining potentially useful signals such as sentiment intensity and structured patterns
- A small number of very short texts (e.g., "Ok", "Yup", numeric values like "645") remain, representing low-information conversational messages that may be filtered in later stages
- The overall text length distribution remains consistent after cleaning, with only slight increases due to contraction expansion
- No significant reduction in text length was observed, confirming that only structural noise is removed while preserving core semantic content

Overall, the cleaning process is controlled, non-aggressive, and preserves both the statistical and semantic integrity of the dataset

### Duplicate Overview

In [25]:
dup_count = df.duplicated(subset="clean_text").sum()
print("Duplicate rows:", dup_count)

Duplicate rows: 415


**Observation: Duplicate Texts**
- 415 duplicate rows are present based on the cleaned text column, indicating that identical messages appear multiple times after normalization
- If not handled, these duplicates can bias the model by over-representing certain samples

### Inspect Duplicate Samples

In [26]:
df_dups = df[df.duplicated(subset="clean_text", keep=False)]
df_dups.sample(5, random_state=RANDOM_STATE)[[TARGET_COL, TEXT_COL, "clean_text"]]

,label,text,clean_text
3681,ham,I cant pick the phone right now. Pls send a me...,I cannot pick the phone right now. Pls send a ...
539,ham,Ummmmmaah Many many happy returns of d day my ...,Ummmmmaah Many many happy returns of d day my ...
522,ham,Shall i come to get pickle,Shall i come to get pickle
1571,ham,No:-)i got rumour that you going to buy apartm...,No:-)i got rumour that you going to buy apartm...
2379,ham,"Hi, Mobile no. &lt;#&gt; has added you in th...","Hi, Mobile no. <#> has added you in their cont..."


In [27]:
# Label consistency check
label_conflicts = df_dups.groupby("clean_text")[TARGET_COL].nunique()
conflicts = (label_conflicts > 1).sum()
print("Duplicate texts with conflicting labels:", conflicts)

Duplicate texts with conflicting labels: 0


**Observation: Duplicate Samples**
- Duplicate rows correspond to identical cleaned text, confirming that repeated messages exist in the dataset
- Sample inspection shows that duplicates are consistent in content and formatting after cleaning
- No label inconsistencies are observed across duplicate texts (0 conflicts), indicating redundancy rather than labeling issues

### Remove Duplicates

In [28]:
df_before = len(df)

df = df.drop_duplicates(subset="clean_text", keep="first").copy()
df_after = len(df)

### Compare Before vs After

In [29]:
removed_pct = (df_before - df_after) / df_before
print("Rows before:", df_before)
print("Rows after:", df_after)
print("Removed %:", f"{removed_pct:.2%}")

Rows before: 5572
Rows after: 5157
Removed %: 7.45%


**Observation: Duplicate Removal Impact**
- 415 rows (~7.45%) were removed after deduplication based on cleaned text
- This indicates a moderate level of redundancy in the dataset
- Removing duplicates helps prevent over-representation of repeated messages and reduces potential bias in model training

### Validation

In [30]:
remaining_dups = df.duplicated(subset="clean_text").sum()
print("Remaining duplicates:", remaining_dups)

Remaining duplicates: 0


**Observation: Duplicate Removal Validation**
- No duplicate rows remain based on the cleaned text column after deduplication
- This confirms that duplicate removal was successful and the dataset is free from repeated samples
- Deduplication ensures that each unique message is represented only once, reducing redundancy and potential bias in downstream modeling

### Finalize cleaned dataset

#### 1. Select Relevant Columns

In [31]:
# Keep relevant columns for modeling and reference
final_df = df[[TARGET_COL, TEXT_COL, "clean_text"]].copy()
final_df.head(2)

,label,text,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...","Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif you oni...


#### 2. Rename Columns

In [32]:
final_df = final_df.rename(columns={
    TARGET_COL: "label",
    TEXT_COL: "raw_text",
    "clean_text": "text"
})
final_df.head(2)

,label,raw_text,text
0,ham,"Go until jurong point, crazy.. Available only ...","Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif you oni...


#### 3. Reset Index

In [33]:
final_df = final_df.reset_index(drop=True)
final_df.head(2)

,label,raw_text,text
0,ham,"Go until jurong point, crazy.. Available only ...","Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif you oni...


#### 4. Final Dataset Overview

In [34]:
print("Final shape:", final_df.shape)
final_df.head()

Final shape: (5157, 3)


,label,raw_text,text
0,ham,"Go until jurong point, crazy.. Available only ...","Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif you oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...,YOU dun say so early hor... YOU c already then...
4,ham,"Nah I don't think he goes to usf, he lives aro...","Nah I do not think he goes to usf, he lives ar..."


In [35]:
# Class distribution
final_df["label"].value_counts(normalize=True)

label
ham     0.875703
spam    0.124297
Name: proportion, dtype: float64

#### 5. Validation Checks

In [36]:
# Null check
print("Nulls in text:", final_df["text"].isna().sum())

# Duplicate check
print("Duplicate rows:", final_df.duplicated(subset="text").sum())

Nulls in text: 0
Duplicate rows: 0


**Observation: Final Dataset**
- The dataset has been reduced to relevant columns required for modeling while retaining raw text for reference
- No null values or duplicate entries are present in the cleaned text column
- Class distribution remains imbalanced (ham ~87.6%, spam ~12.4%), which should be considered during model evaluation and training
- The dataset is clean, structured, and ready for downstream NLP tasks such as vectorization and modeling

### Save Processed Dataset

In [37]:
# Create directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save processed dataset
final_df.to_csv(PROCESSED_DATA_PATH, index=False)

print("Saved to:", PROCESSED_DATA_PATH)

Saved to: C:\Users\amiti\Documents\Data_Science_Prep\Career_Switch\nlp-playbook\data\processed\cleaned_sms.csv


**Observation: Data Persistence**
- The cleaned dataset is saved locally for reuse in downstream tasks such as feature engineering and modeling
- This improves reproducibility and avoids re-running preprocessing steps
- Raw data is sourced from an external URL, while processed data is stored within the repository for consistent access